# 지금까지 구한 값들로 유튜버의 위험도를 계산하겠습니다.

```
최종 위험도 (ABCDF)
├── 평판 위험도 33%    민감 키워드 매칭도 
├── 트래픽 위험도 33%  이탈 확률 + 정체 확률 + 조회수 변동성 + 업로드 주기 불확실성
└── 팬덤 위험도 33%   활성 시청자 비율 
```

In [5]:
#라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing  import StandardScaler
from sklearn.cluster        import KMeans
from sklearn.metrics        import silhouette_score

RANDOM_STATE = 42
print("라이브러리 로드 완료")


라이브러리 로드 완료


In [6]:
# 폰트
import matplotlib.pyplot as plt

# macOS에서 가장 많이 사용하는 설정
plt.rcParams["font.family"] = "AppleGothic" # 혹은 "Apple SD Gothic Neo"
plt.rcParams["axes.unicode_minus"] = False

print("macOS 한글 폰트 설정 완료")

macOS 한글 폰트 설정 완료


# 1. 데이터 불러오기

In [ ]:
# 민감 키워드 매칭도
sensitive_keyword_df = pd.read_csv("../../data/raw/sensitive_keyword_score.csv")

# 이탈 확률
churn_prob_df = pd.read_csv("../../data/raw/churn_probability_output.csv")

# 정체 확률
stagnation_prob_df = pd.read_csv("../../data/raw/stagnation_probability_output.csv")

# 조회수 변동성
view_volatility_df = pd.read_csv("../../data/raw/view_volatility_output.csv")

# 업로드 주기
upload_regularity_df = pd.read_csv("../../data/raw/upload_regularity_score.csv")

# 활성 시청자
active_viewer_df = pd.read_csv("../../data/raw/active_viewer_score.csv")


## 1-1 이탈 확률 필터링

In [8]:
# channel_id 기준으로 dataset_long 필터링

churn_prob_df = churn_prob_df[

    churn_prob_df["channel_id"].isin(

        sensitive_keyword_df["channel_id"]

    )

]

churn_prob_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3336 entries, 140 to 8081
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   channel_id    3336 non-null   object 
 1   title         3336 non-null   object 
 2   actual_label  3336 non-null   int64  
 3   prob_lr       3336 non-null   float64
 4   prob_rf       3336 non-null   float64
 5   prob_xgb      3336 non-null   float64
 6   churn_prob    3336 non-null   float64
dtypes: float64(4), int64(1), object(2)
memory usage: 208.5+ KB


In [9]:
sensitive_keyword_df.head()

,channel_id,channel_title,sensitive_score,n_sensitive_videos,sensitive_video_ratio,cat_politics,cat_hate,cat_aggro,cat_adult_illegal
0,UCdf4di4k4fU5lio8ZXQx6mQ,루스명품,0.7733,37,0.74,0,0,2,37
1,UC6nxfRgvAEcUPxQxsvF6lzw,미친감성 (감성사운드),0.7000,47,0.94,1,0,52,0
2,UCWLdJ-g9j-ZUwlnR7fU1yCQ,강스라이팅,0.6933,29,0.58,43,0,5,0
3,UCzz58-H0wzmGoFi3MO_9ePQ,온마이크,0.6733,32,0.64,46,0,4,0
4,UCHkaQ4y4j3zrB0FfTU1Lc9w,더빙신안윤상,0.6267,32,0.64,45,0,1,1


In [10]:
churn_prob_df.head()

,channel_id,title,actual_label,prob_lr,prob_rf,prob_xgb,churn_prob
140,UC347yXLUujTh4uCFwHBgdWA,바라봐라,1,0.978,0.954,1.0,0.977
147,UCVFSrUkV0LTzi4bLICZ7SEw,숏쯔,1,0.978,0.954,1.0,0.977
291,UCp57m_15RPFgvjsx3FrCE_A,HANKOOK NORE,1,0.983,0.923,1.0,0.969
338,UCbsT8D_yt_Rvhk94zqTkx0w,Snow Toy TV [스노우 토이],1,0.962,0.934,1.0,0.965
416,UCE1N1MU4TwiEVrqJY6UW7Hw,솔라시도,1,0.939,0.946,1.0,0.961


In [11]:
stagnation_prob_df.head()

,channel_id,title,actual_label,prob_lr,prob_rf,prob_xgb,stagnation_prob
0,UCklDsqQMGPYIMpIhQ5Jz6jA,모양몬 로블록스,1,1.0,0.988,1.0,0.996
1,UC9gxOp_-R78phMHmv2bW_sg,원지의하루,1,1.0,0.988,1.0,0.996
2,UCzjEp3u8RMkOJwpc3RzxmWg,콩콩이다,1,1.0,0.988,1.0,0.996
3,UCyyWcEhqfaqJQ56XdA44RYQ,챔보,1,1.0,0.988,1.0,0.996
4,UCZyYOvUGGiDFkEB8G9StWBA,집에서안나와 - with ANNA,1,1.0,0.988,1.0,0.996


In [12]:
view_volatility_df.head()

,channel_id,title,actual_label,prob_lr,prob_rf,prob_xgb,volatility_prob
0,UC0bXUn25faBH-JzlUEHNh7g,웃긴협회,1,1.0,0.991,1.0,0.997
1,UCdUlCaxi7gx9Q-WDVDe30YA,배스킨라빈스,1,1.0,0.991,1.0,0.997
2,UCxXgIeE5hxWxHG6dz9Scg2w,RAIN's Official Channel,1,1.0,0.990,1.0,0.997
3,UCCn0jN1hH5m6hMOB51WyAKw,세호가중계,1,1.0,0.990,1.0,0.997
4,UC3pk0eDcZx_oYGCWNyJpjMQ,나돌_NADOL,1,1.0,0.990,1.0,0.997


In [13]:
upload_regularity_df.tail()

,channel_id,channel_title,regularity_score,mean_interval_days,std_interval_days,max_gap_days,cv,outlier_ratio,gap_ratio
3328,UCCJkwrmhIqWkSFV-sQol4Qw,밍꼬발랄Mingggo,0.3586,30.1,202.2,1431.0,6.727,0.0204,47.60
3329,UCGNhjyXPRc3IX308JVxeZ0g,Chiro 치로와 친구들,0.3581,32.7,221.8,1569.0,6.787,0.0204,48.02
3330,UCSuhLNKQsW0-_JtjzIumbvA,차살때 스튜디오,0.3578,8.2,55.7,394.0,6.822,0.0204,48.26
3331,UCObpcVC0xLXzyJr1SlKrheA,박축공 Football Park,0.3578,5.3,36.2,256.0,6.820,0.0204,48.25
3332,UC5HpbZB-E-DzFpWZmVlcV9A,KBS 해피투게더,0.3577,10.9,74.5,527.0,6.836,0.0204,48.36


In [14]:
active_viewer_df.head()

,channel_id,title,subscriber_count,avg_view_count,view_per_sub,cluster,cluster_name,active_viewer_score
0,UCy2bCk5KnIfjmWYFHMZcd5w,RosaliaVEVO,137000,30169503.76,220.215356,1,슈퍼팬 (최고 활성),0.9518
1,UCqPbkJrjrsQb_RGlC_4HEgg,TheKidLAROIVEVO,102000,4736267.86,46.433999,1,슈퍼팬 (최고 활성),0.9265
2,UCSfoU2V_ECZEqnuNjSZwkTw,timeory,158000,3207053.22,20.297805,1,슈퍼팬 (최고 활성),0.9117
3,UCmvqXRPTfMgkKF3MFWylMqQ,HITEJINRO,125000,2303630.72,18.429046,1,슈퍼팬 (최고 활성),0.9051
4,UCq69tzvGPKrnbdzrDJmu7BQ,겨울아빠 WinterPapa,281000,4345090.68,15.462956,1,슈퍼팬 (최고 활성),0.9028


In [15]:
import pandas as pd

# 1. 모든 데이터프레임의 기준 컬럼을 인덱스로 설정합니다.
dfs = [
    df.set_index("channel_id")
    for df in [
        sensitive_keyword_df,
        churn_prob_df,
        stagnation_prob_df,
        view_volatility_df,
        upload_regularity_df,
        active_viewer_df,
    ]
]

# 2. axis=1(열 방향)로 outer join(기본값)합니다.
# 3. reset_index()를 통해 channel_id를 다시 컬럼으로 빼줍니다.
risk_df = pd.concat(dfs, axis=1).reset_index()
risk_df.head()

,channel_id,channel_title,sensitive_score,n_sensitive_videos,sensitive_video_ratio,cat_politics,cat_hate,cat_aggro,cat_adult_illegal,title,...,cv,outlier_ratio,gap_ratio,title,subscriber_count,avg_view_count,view_per_sub,cluster,cluster_name,active_viewer_score
0,UCdf4di4k4fU5lio8ZXQx6mQ,루스명품,0.7733,37,0.74,0,0,2,37,루스명품,...,0.739,0.1020,3.04,루스명품,131000,36940.94,0.281992,0,일반 팬 (건강),0.6058
1,UC6nxfRgvAEcUPxQxsvF6lzw,미친감성 (감성사운드),0.7000,47,0.94,1,0,52,0,미친감성 (감성사운드),...,1.488,0.0408,7.72,미친감성 (감성사운드),142000,45643.04,0.321430,3,소극적 팬 (참여 저조),0.3650
2,UCWLdJ-g9j-ZUwlnR7fU1yCQ,강스라이팅,0.6933,29,0.58,43,0,5,0,강스라이팅,...,1.773,0.0408,11.00,강스라이팅,166000,887.62,0.005347,3,소극적 팬 (참여 저조),0.3685
3,UCzz58-H0wzmGoFi3MO_9ePQ,온마이크,0.6733,32,0.64,46,0,4,0,온마이크,...,1.613,0.0816,6.47,온마이크,122000,8637.88,0.070802,3,소극적 팬 (참여 저조),0.3582
4,UCHkaQ4y4j3zrB0FfTU1Lc9w,더빙신안윤상,0.6267,32,0.64,45,0,1,1,더빙신안윤상,...,1.226,0.1020,5.15,더빙신안윤상,219000,29263.24,0.133622,3,소극적 팬 (참여 저조),0.3626


In [16]:
risk_df = risk_df[['channel_id', 'channel_title', 'sensitive_score', 'churn_prob', 'stagnation_prob', 'volatility_prob', 'regularity_score','active_viewer_score']]
risk_df.head()

,channel_id,channel_title,channel_title,sensitive_score,churn_prob,stagnation_prob,volatility_prob,regularity_score,active_viewer_score
0,UCdf4di4k4fU5lio8ZXQx6mQ,루스명품,루스명품,0.7733,0.066,0.925,0.023,0.7558,0.6058
1,UC6nxfRgvAEcUPxQxsvF6lzw,미친감성 (감성사운드),미친감성 (감성사운드),0.7000,0.144,0.987,0.995,0.5537,0.3650
2,UCWLdJ-g9j-ZUwlnR7fU1yCQ,강스라이팅,강스라이팅,0.6933,0.920,0.026,0.003,0.4681,0.3685
3,UCzz58-H0wzmGoFi3MO_9ePQ,온마이크,온마이크,0.6733,0.873,0.013,0.979,0.5677,0.3582
4,UCHkaQ4y4j3zrB0FfTU1Lc9w,더빙신안윤상,더빙신안윤상,0.6267,0.168,0.017,0.020,0.6324,0.3626


In [17]:
risk_df = risk_df.loc[:, ~risk_df.columns.duplicated()]
risk_df

,channel_id,channel_title,sensitive_score,churn_prob,stagnation_prob,volatility_prob,regularity_score,active_viewer_score
0,UCdf4di4k4fU5lio8ZXQx6mQ,루스명품,0.7733,0.066,0.925,0.023,0.7558,0.6058
1,UC6nxfRgvAEcUPxQxsvF6lzw,미친감성 (감성사운드),0.7000,0.144,0.987,0.995,0.5537,0.3650
2,UCWLdJ-g9j-ZUwlnR7fU1yCQ,강스라이팅,0.6933,0.920,0.026,0.003,0.4681,0.3685
3,UCzz58-H0wzmGoFi3MO_9ePQ,온마이크,0.6733,0.873,0.013,0.979,0.5677,0.3582
4,UCHkaQ4y4j3zrB0FfTU1Lc9w,더빙신안윤상,0.6267,0.168,0.017,0.020,0.6324,0.3626
...,...,...,...,...,...,...,...,...
3331,UCCW6uYIUP4YKKe50k-ktVAw,민또 경또,0.0000,0.004,0.648,0.000,0.6460,0.6061
3332,UCCXeZkv4-v2TlH_J49I_icw,휴복,0.0000,0.045,0.841,0.000,0.5839,0.3690
3333,UCCZ3InIIho_BmoaZIiiwDSg,땅콩찐콩,0.0000,0.003,0.838,0.005,0.7805,0.6087
3334,UCb3XMd6KgVhp48VcJAqEeiw,RE:REVOLUTION [혁명의 심장],0.0000,0.039,0.008,0.989,0.5798,0.3742
